In [1]:
"""
exp12 Step 01: VIX Fear Re-entry Exploration (EXPLORATION 2009-08-13 -> 2014-12-31 ONLY; context, PROTOCOL.md §6)

The 6 candidates of PROTOCOL.md §4 (exits E1 = exp04's x = 0, n = 1 and E2 = exp07's k = 4; modifications M1 fear-fade
re-entry, M2 term-structure re-entry, M3 no exit into fear; _prev VIX timing only) are replayed as ONE continuous path
over 2009-08-13 -> 2014-12-31 (start invested, no forced buy-back; the VIX / VIX3M term feature exists from 2009-08-13),
against buy-and-hold and the unmodified E1 and E2 over the same sessions. SPY bars and every VIX load are cut at
so.config.EXPLORATION_DATA_CUTOFF_DATE_STR (2015-03-18): nothing of the walk-forward period is read.

Context only: the result changes nothing in the protocol. Caveat: 2010-2011 and their VIX peaks are known to everyone.
Outputs: store04_experiments/exp12_vix_fear_reentry/step01_exploration_data/. Trials: 6 (stage "exploration").
"""
None

In [2]:
# IMPORT GENERAL PACKAGES
import pandas as pd
import numpy as np
# IMPORT THE SHARED CONFIGURATION, THE EXPERIMENT CONFIGURATION AND THE PATHS
from so import config, paths
from experiments.exp12_vix_fear_reentry import config as exp_config
# IMPORT DATA LOADING, EXECUTION, DAILY FEATURE AND VIX FEATURE FUNCTIONS
from so.core.raw_data import get_complete_ohlcv_pdf
from so.core.trade_execution import get_ohlcv_array_dict
from so.features.daily_features import get_daily_feature_pdf
from so.features.vix_features import get_vix_daily_feature_dict
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import write_csv_file_to_path
# IMPORT EXIT AND RE-ENTRY SIMULATION FUNCTIONS
from so.core.reentry_simulation import get_buy_and_hold_result_dict
# IMPORT THE CONTINUOUS REPLAY FUNCTIONS
from so.core.continuous_replay import simulate_continuous_replay_dict, get_episode_scorecard_pdf, get_replay_summary_dict
# IMPORT THE EXPERIMENT FUNCTIONS
from experiments.exp12_vix_fear_reentry.rules import get_rule_candidate_list, get_unmodified_candidate_list, get_rule_builder_func, get_vix_scorecard_pdf
# IMPORT TRIAL LOG FUNCTIONS
from so.core.trial_log import get_config_hash_str, log_trial_pdf, read_trial_log_pdf, get_trial_count_pdf, print_previous_trial_warning

In [3]:
# CHECK THE REAL DATA ONLY OUTSIDE THE SYNTHETIC SMOKE RUN (scripts/smoke_notebook.py)
CHECK_REAL_DATA_BOOL = __name__ != "__smoke__"
# DISPLAY THE EXPERIMENT, THE PROTOCOL VERSION AND THE CONFIGURATION HASH (SHARED + EXPERIMENT CONFIG)
print(f"Experiment:\t{exp_config.EXPERIMENT_NAME}\tProtocol:\t{exp_config.PROTOCOL_VERSION}\tConfig Hash:\t{get_config_hash_str(exp_config)}")
# DEFINE THE OUTPUT FOLDER OF THIS STEP
output_path_str = paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step01_exploration_data")
print(f"Output folder:\t{output_path_str}")
# CALL FUNCTION TO WARN ABOUT EARLIER ENTRIES OF THIS STAGE (A RERUN ADDS TRIALS)
previous_trial_count = print_previous_trial_warning(exp_config, "exploration")

Experiment:	exp12_vix_fear_reentry	Protocol:	1.0	Config Hash:	eec3977199
Output folder:	C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp12_vix_fear_reentry/step01_exploration_data/
ℹ️ No earlier 'exploration' entry of exp12_vix_fear_reentry in the trial log (config hash eec3977199).


In [4]:
# CALL FUNCTION TO READ THE MINUTE BARS UP TO THE EXPLORATION DATA CUTOFF (NOTHING FROM THE WALK-FORWARD PERIOD IS READ)
cutoff_date_str = exp_config.EXPLORATION_DATA_CUTOFF_DATE_STR
complete_ohlcv_pdf = get_complete_ohlcv_pdf(cutoff_date_str_in=cutoff_date_str)
assert complete_ohlcv_pdf["date"].max() <= pd.Timestamp(cutoff_date_str).date()
# CALL FUNCTION TO CONVERT THE DATA TO ARRAYS AND BUILD THE DAILY TABLE
ohlcv_array_dict = get_ohlcv_array_dict(complete_ohlcv_pdf)
daily_pdf = get_daily_feature_pdf(ohlcv_array_dict)
# CALL FUNCTION TO LOAD THE VIX AND VIX3M DAILY BARS WITH THE SAME CUTOFF AND BUILD THE _prev FEATURES
vix_dict = get_vix_daily_feature_dict(daily_pdf, cutoff_date_str, [exp_config.VIX_VARIANT_STR])
vix_feature_pdf = vix_dict["feature_pdf"]
assert all(last_date <= pd.Timestamp(cutoff_date_str).date() for last_date in vix_dict["last_date_dict"].values())
# DEFINE THE EXPLORATION WINDOW AND ONE PERIOD PER CALENDAR YEAR (FOR THE YEARS-WON COUNT)
exploration_start_str, exploration_end_str = exp_config.EXPLORATION_REPLAY_START_DATE_STR, exp_config.EXPLORATION_REPLAY_END_DATE_STR
session_date_arr = np.array(ohlcv_array_dict["session_date_list"])
window_date_arr = session_date_arr[(session_date_arr >= pd.Timestamp(exploration_start_str).date()) & (session_date_arr <= pd.Timestamp(exploration_end_str).date())]
if CHECK_REAL_DATA_BOOL:
    assert str(window_date_arr[0]) == exploration_start_str and str(window_date_arr[-1]) == exploration_end_str
year_period_pdf = pd.DataFrame([{"period_id": year, "period_start": min(d for d in window_date_arr if d.year == year), "period_end": max(d for d in window_date_arr if d.year == year)}
                                for year in sorted({d.year for d in window_date_arr})])
# DISPLAY THE LAST DATES LOADED AND THE WINDOW
print(f"Last SPY date loaded:\t{daily_pdf['date'].max()}\t({len(daily_pdf):,} sessions from {daily_pdf['date'].min()})")
for key_str, last_date in vix_dict["last_date_dict"].items():
    print(f"Last date loaded:\t{key_str}\t{last_date}")
print(f"Replay:\t{window_date_arr[0]} -> {window_date_arr[-1]}\t({len(window_date_arr):,} sessions, {len(year_period_pdf)} years)")
# DISPLAY THE COVERAGE OF THE THREE FEATURES IN THE WINDOW
window_feature_pdf = vix_feature_pdf[vix_feature_pdf["date"].isin(set(window_date_arr))]
print("Missing values in the window:", {col: int(window_feature_pdf[col].isna().sum()) for col in [exp_config.VIX_LEVEL_COL_STR, exp_config.VIX_TERM_COL_STR, exp_config.VIX_PCT250_COL_STR]})
print(f"Share of window decisions with vix_term_prev >= 1:\t{(window_feature_pdf[exp_config.VIX_TERM_COL_STR] >= 1).mean():.3f}"
      f"\twith vix_pct250_prev >= 0.9:\t{(window_feature_pdf[exp_config.VIX_PCT250_COL_STR] >= 0.9).mean():.3f}")

Last SPY date loaded:	2015-03-18	(2,568 sessions from 2005-01-03)
Last date loaded:	vix_daily	2015-03-18
Last date loaded:	vix3m_daily	2015-03-18
Replay:	2009-08-13 -> 2014-12-31	(1,356 sessions, 6 years)
Missing values in the window: {'vix_level_prev': 0, 'vix_term_prev': 0, 'vix_pct250_prev': 0}
Share of window decisions with vix_term_prev >= 1:	0.077	with vix_pct250_prev >= 0.9:	0.086


In [5]:
"""
The 6 candidates and the 2 unmodified exits, each one continuous path (start invested, no forced buy-back), against
buy-and-hold over the same sessions.
"""
# SIMULATE BUY AND HOLD
buy_hold_dict = get_buy_and_hold_result_dict(ohlcv_array_dict, exploration_start_str, exploration_end_str)
print(f"Buy & hold:\ttotal return {buy_hold_dict['metric_dict']['total_return']:.2%} | Sharpe {buy_hold_dict['metric_dict']['sharpe_ratio']:.2f} | max drawdown {buy_hold_dict['metric_dict']['max_drawdown']:.2%}")
# DEFINE THE RULE BUILDER
rule_builder_func = get_rule_builder_func(daily_pdf, vix_feature_pdf)
# LISTS TO HOLD THE RESULTS
row_dict_list, scorecard_pdf_list = [], []
# ITERATE OVER THE UNMODIFIED EXITS (BASELINES, NOT LOGGED) AND THE 6 CANDIDATES
for candidate_dict in get_unmodified_candidate_list() + get_rule_candidate_list():
    # BUILD AND REPLAY THE CANDIDATE
    exit_signal_arr, reentry_func, reason_str = rule_builder_func(candidate_dict)
    simulation_dict = simulate_continuous_replay_dict(ohlcv_array_dict, exploration_start_str, exploration_end_str, exp_config.UNTOUCHED_START_DATE_STR,
                                                      volatility_arr_in=np.full(len(daily_pdf), np.nan), stop_k_in=None, reentry_func_in=reentry_func,
                                                      reentry_reason_str_in=reason_str, exit_signal_arr_in=exit_signal_arr, max_cash_sessions_in=exp_config.MAX_CASH_SESSIONS)
    # SUMMARIZE THE PATH (YEARS AS PERIODS) AND BUILD ITS SCORECARD
    summary_dict = get_replay_summary_dict(simulation_dict["daily_equity_pdf"], buy_hold_dict["daily_equity_pdf"], year_period_pdf, exp_config.BOOTSTRAP_BLOCK_MONTH_COUNT)
    scorecard_pdf = get_vix_scorecard_pdf(get_episode_scorecard_pdf(simulation_dict, ohlcv_array_dict), simulation_dict, vix_feature_pdf)
    scorecard_pdf_list.append(scorecard_pdf.assign(**candidate_dict))
    # STORE THE ROW
    trigger_series = scorecard_pdf["reentry_trigger"] if len(scorecard_pdf) else pd.Series(dtype=str)
    row_dict_list.append({**candidate_dict, "total_return": summary_dict["strategy_total_return"], "buy_hold_total_return": summary_dict["buy_hold_total_return"],
                          "excess_total_return": summary_dict["excess_total_return"], "annualized_return": summary_dict["strategy_annualized_return"],
                          "sharpe_ratio": summary_dict["strategy_sharpe_ratio"], "max_drawdown": summary_dict["strategy_max_drawdown"], "in_market_pct": summary_dict["in_market_pct"],
                          "exit_count": simulation_dict["metric_dict"]["signal_exit_count"], "mean_cash_sessions": simulation_dict["metric_dict"]["mean_cash_sessions"],
                          "bought_back_lower_pct": simulation_dict["metric_dict"]["positive_episode_pct"],
                          "product_s_over_r": float(np.prod(scorecard_pdf["s_over_r"])) if len(scorecard_pdf) else 1.0,
                          "vix_reentry_count": int(trigger_series.isin(["M1", "M2"]).sum()), "original_reentry_count": int((trigger_series == "original rule").sum()),
                          "years_won": summary_dict["period_windows_beating_buy_hold"], "year_count": summary_dict["period_window_count"],
                          "annualized_log_excess": summary_dict["log_excess_annualized_log_excess"], "log_excess_ci_low": summary_dict["log_excess_ci_low"], "log_excess_ci_high": summary_dict["log_excess_ci_high"]})
# CONVERT THE RESULTS AND ADD THE COMPARISON WITH THE UNMODIFIED EXIT OF EACH CANDIDATE
all_row_pdf = pd.DataFrame(row_dict_list)
unmodified_total_dict = all_row_pdf[all_row_pdf["modification"] == exp_config.UNMODIFIED_STR].set_index("exit_rule")["total_return"].to_dict()
all_row_pdf["unmodified_total_return"] = all_row_pdf["exit_rule"].map(unmodified_total_dict)
all_row_pdf["excess_vs_unmodified"] = all_row_pdf["total_return"] - all_row_pdf["unmodified_total_return"]
exploration_pdf = all_row_pdf[all_row_pdf["modification"] != exp_config.UNMODIFIED_STR].reset_index(drop=True)
exploration_scorecard_pdf = pd.concat(scorecard_pdf_list, ignore_index=True)
# PREVIEW DATAFRAME
all_row_pdf[["candidate_name", "total_return", "buy_hold_total_return", "excess_total_return", "unmodified_total_return", "excess_vs_unmodified", "in_market_pct",
             "exit_count", "vix_reentry_count", "original_reentry_count", "product_s_over_r", "max_drawdown", "annualized_log_excess", "log_excess_ci_low", "log_excess_ci_high"]].round(4)

Buy & hold:	total return 104.57% | Sharpe 0.92 | max drawdown -19.44%


,candidate_name,total_return,buy_hold_total_return,excess_total_return,unmodified_total_return,excess_vs_unmodified,in_market_pct,exit_count,vix_reentry_count,original_reentry_count,product_s_over_r,max_drawdown,annualized_log_excess,log_excess_ci_low,log_excess_ci_high
0,E1none,0.4900,1.0457,-0.5557,0.4900,0.0000,0.8694,16,0,16,0.7291,-0.2049,-0.0585,-0.1311,-0.0035
1,E2none,0.6114,1.0457,-0.4344,0.6114,0.0000,0.9196,8,0,8,0.7879,-0.2083,-0.0441,-0.1087,-0.0010
2,E1M1,1.0451,1.0457,-0.0007,0.4900,0.5551,0.9454,16,9,7,1.0008,-0.1405,-0.0001,-0.0376,0.0525
3,E1M2,0.7125,1.0457,-0.3332,0.4900,0.2225,0.9151,16,5,11,0.8381,-0.1635,-0.0328,-0.0750,0.0004
4,E1M3,0.5198,1.0457,-0.5259,0.4900,0.0298,0.9018,14,0,14,0.7437,-0.2090,-0.0549,-0.1284,-0.0028
5,E2M1,0.8425,1.0457,-0.2032,0.6114,0.2312,0.9690,10,6,4,0.9013,-0.1955,-0.0193,-0.0527,0.0122
6,E2M2,0.6638,1.0457,-0.3819,0.6114,0.0525,0.9483,11,5,6,0.8141,-0.1847,-0.0381,-0.0879,-0.0062
7,E2M3,0.7476,1.0457,-0.2982,0.6114,0.1362,0.9646,7,0,7,0.8546,-0.1944,-0.0291,-0.0665,-0.0031


In [6]:
"""
Episode scorecard of every path. S / R > 1 = bought back lower than sold; reentry_trigger = M1, M2, original rule or end
of data; vix_at_exit / vix_at_reentry = vix_level_prev at the two decisions.
"""
# DISPLAY THE SCORECARDS
scorecard_col_list = ["exit_date", "exit_fill_price", "reentry_date", "reentry_fill_price", "s_over_r", "sessions_out", "reentry_trigger", "vix_at_exit", "vix_at_reentry",
                      "buy_hold_return_out", "max_decline_out_pct", "open_at_end"]
for candidate_name in all_row_pdf["candidate_name"]:
    candidate_scorecard_pdf = exploration_scorecard_pdf[exploration_scorecard_pdf["candidate_name"] == candidate_name] if len(exploration_scorecard_pdf) else exploration_scorecard_pdf
    print(f"\n{candidate_name}:\t{len(candidate_scorecard_pdf)} episodes, {int((candidate_scorecard_pdf['s_over_r'] > 1).sum()) if len(candidate_scorecard_pdf) else 0} bought back lower")
    if len(candidate_scorecard_pdf):
        display(candidate_scorecard_pdf[scorecard_col_list].round(4))


E1none:	16 episodes, 0 bought back lower


,exit_date,exit_fill_price,reentry_date,reentry_fill_price,s_over_r,sessions_out,reentry_trigger,vix_at_exit,vix_at_reentry,buy_hold_return_out,max_decline_out_pct,open_at_end
0,2010-05-20,107.65,2010-05-27,110.70,0.9724,5,original rule,34.95,34.20,0.0281,-0.0305,False
1,2010-05-28,109.45,2010-06-03,110.78,0.9880,3,original rule,29.61,30.05,0.0120,-0.0191,False
2,2010-06-04,106.76,2010-06-15,112.02,0.9530,7,original rule,29.70,28.68,0.0491,-0.0199,False
3,2010-06-22,109.50,2010-07-26,111.63,0.9809,23,original rule,25.22,23.65,0.0193,-0.0765,False
4,2010-07-27,111.52,2010-08-02,112.73,0.9893,4,original rule,22.75,23.57,0.0107,-0.0229,False
5,2010-08-11,109.27,2010-09-13,112.74,0.9692,22,original rule,22.46,21.94,0.0316,-0.0457,False
6,2011-08-02,125.53,2011-10-27,128.47,0.9771,61,original rule,23.33,30.02,0.0233,-0.1443,False
7,2011-10-31,125.56,2011-11-08,127.89,0.9818,6,original rule,24.49,29.95,0.0184,-0.0323,False
8,2011-11-09,123.20,2011-12-07,126.65,0.9728,19,original rule,27.63,28.13,0.0278,-0.0559,False
9,2011-12-08,123.80,2011-12-23,126.37,0.9797,11,original rule,28.67,21.16,0.0206,-0.0305,False



E2none:	8 episodes, 0 bought back lower


,exit_date,exit_fill_price,reentry_date,reentry_fill_price,s_over_r,sessions_out,reentry_trigger,vix_at_exit,vix_at_reentry,buy_hold_return_out,max_decline_out_pct,open_at_end
16,2010-05-25,107.76,2010-06-21,111.30,0.9682,18,original rule,38.24,23.77,0.0327,-0.0290,False
17,2011-08-05,120.30,2011-10-27,128.47,0.9364,58,original rule,31.75,30.02,0.0677,-0.1071,False
18,2011-11-28,119.60,2011-12-07,126.65,0.9443,7,original rule,34.18,28.13,0.0588,-0.0003,False
19,2011-12-08,123.80,2011-12-23,126.37,0.9797,11,original rule,28.67,21.16,0.0206,-0.0305,False
20,2011-12-28,124.75,2012-01-04,127.69,0.9770,4,original rule,21.91,22.97,0.0234,-0.0001,False
21,2012-06-01,128.29,2012-06-05,129.07,0.9940,2,original rule,24.06,26.12,0.0059,-0.0090,False
22,2012-06-28,132.87,2012-07-02,136.53,0.9732,2,original rule,19.45,17.08,0.0274,-0.0008,False
23,2012-11-08,138.17,2012-11-19,139.01,0.9940,7,original rule,19.08,16.41,0.0059,-0.0252,False



E1M1:	16 episodes, 4 bought back lower


...

,exit_date,exit_fill_price,reentry_date,reentry_fill_price,s_over_r,sessions_out,reentry_trigger,vix_at_exit,vix_at_reentry,buy_hold_return_out,max_decline_out_pct,open_at_end
70,2010-05-25,107.76,2010-05-28,109.47,0.9844,3,M1,38.24,29.61,0.0157,-0.0085,False
71,2010-06-04,106.76,2010-06-11,109.66,0.9736,5,M1,29.70,30.39,0.0270,-0.0199,False
72,2011-08-05,120.30,2011-08-10,112.15,1.0727,3,M1,31.75,35.12,-0.0679,-0.0835,False
73,2011-09-22,113.01,2011-10-06,116.61,0.9691,10,M1,36.78,37.64,0.0317,-0.0495,False
74,2011-11-28,119.60,2011-12-01,124.89,0.9576,3,M1,34.18,27.80,0.0441,-0.0003,False
75,2011-12-08,123.80,2011-12-13,123.09,1.0058,3,M1,28.67,25.67,-0.0059,-0.0110,False
76,2011-12-28,124.75,2012-01-04,127.69,0.9770,4,original rule,21.91,22.97,0.0234,-0.0001,False
77,2012-06-01,128.29,2012-06-05,129.07,0.9940,2,original rule,24.06,26.12,0.0059,-0.0090,False
78,2012-06-28,132.87,2012-07-02,136.53,0.9732,2,original rule,19.45,17.08,0.0274,-0.0008,False
79,2012-11-08,138.17,2012-11-19,139.01,0.9940,7,original rule,19.08,16.41,0.0059,-0.0252,False



E2M2:	11 episodes, 1 bought back lower


,exit_date,exit_fill_price,reentry_date,reentry_fill_price,s_over_r,sessions_out,reentry_trigger,vix_at_exit,vix_at_reentry,buy_hold_return_out,max_decline_out_pct,open_at_end
80,2010-05-25,107.76,2010-05-26,107.16,1.0056,1,M2,38.24,34.93,-0.0058,-0.0085,False
81,2010-05-28,109.45,2010-06-03,110.78,0.9880,3,M2,29.61,30.05,0.0120,-0.0191,False
82,2010-06-04,106.76,2010-06-10,109.21,0.9776,4,M2,29.70,33.73,0.0228,-0.0199,False
83,2011-08-05,120.30,2011-09-01,120.87,0.9953,19,M2,31.75,31.78,0.0046,-0.0835,False
84,2011-09-22,113.01,2011-10-06,116.61,0.9691,10,M2,36.78,37.64,0.0317,-0.0495,False
85,2011-11-28,119.60,2011-12-07,126.65,0.9443,7,original rule,34.18,28.13,0.0588,-0.0003,False
86,2011-12-08,123.80,2011-12-23,126.37,0.9797,11,original rule,28.67,21.16,0.0206,-0.0305,False
87,2011-12-28,124.75,2012-01-04,127.69,0.9770,4,original rule,21.91,22.97,0.0234,-0.0001,False
88,2012-06-01,128.29,2012-06-05,129.07,0.9940,2,original rule,24.06,26.12,0.0059,-0.0090,False
89,2012-06-28,132.87,2012-07-02,136.53,0.9732,2,original rule,19.45,17.08,0.0274,-0.0008,False



E2M3:	7 episodes, 0 bought back lower


,exit_date,exit_fill_price,reentry_date,reentry_fill_price,s_over_r,sessions_out,reentry_trigger,vix_at_exit,vix_at_reentry,buy_hold_return_out,max_decline_out_pct,open_at_end
91,2010-05-28,109.45,2010-06-21,111.30,0.9834,15,original rule,29.61,23.77,0.0167,-0.0439,False
92,2011-11-28,119.60,2011-12-07,126.65,0.9443,7,original rule,34.18,28.13,0.0588,-0.0003,False
93,2011-12-08,123.80,2011-12-23,126.37,0.9797,11,original rule,28.67,21.16,0.0206,-0.0305,False
94,2011-12-28,124.75,2012-01-04,127.69,0.9770,4,original rule,21.91,22.97,0.0234,-0.0001,False
95,2012-06-01,128.29,2012-06-05,129.07,0.9940,2,original rule,24.06,26.12,0.0059,-0.0090,False
96,2012-06-28,132.87,2012-07-02,136.53,0.9732,2,original rule,19.45,17.08,0.0274,-0.0008,False
97,2012-11-08,138.17,2012-11-19,139.01,0.9940,7,original rule,19.08,16.41,0.0059,-0.0252,False


In [7]:
# SAVE THE RESULTS
write_csv_file_to_path(all_row_pdf, f"{output_path_str}exploration_candidate_data.csv", "W")
write_csv_file_to_path(exploration_scorecard_pdf, f"{output_path_str}exploration_scorecard_data.csv", "W")

ℹ️ Saving In Overwrite Mode.	(💾 In Local File Storage)
⚠️ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp12_vix_fear_reentry/step01_exploration_data/exploration_candidate_data.csv' Not Found!	(💾 In Local File Storage)
✅ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp12_vix_fear_reentry/step01_exploration_data/exploration_candidate_data.csv' Has Been Saved!
ℹ️ Saving In Overwrite Mode.	(💾 In Local File Storage)
⚠️ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp12_vix_fear_reentry/step01_exploration_data/exploration_scorecard_data.csv' Not Found!	(💾 In Local File Storage)
✅ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp12_vix_fear_reentry/step01_exploration_data/exploration_scorecard_data.csv' Has Been Saved!


In [8]:
# CHECK THE TRIAL BUDGET BEFORE LOGGING (EVERY STAGE OF THIS EXPERIMENT)
trial_log_pdf = read_trial_log_pdf()
experiment_trial_count = int((trial_log_pdf["experiment"] == exp_config.EXPERIMENT_NAME).sum()) if len(trial_log_pdf) else 0
assert experiment_trial_count + len(exploration_pdf) <= exp_config.TRIAL_BUDGET, "❌ Trial budget exceeded"
# LOG THE 6 CANDIDATES AS EXPLORATION TRIALS (ONE WRITE)
exploration_trial_pdf = log_trial_pdf(exp_config, "exploration", exploration_pdf.assign(window=f"{exploration_start_str}->{exploration_end_str}"), ["exit_rule", "modification", "window"],
                                      [col for col in exploration_pdf.columns if col not in ["exit_rule", "modification", "candidate_name", "tie_order"]], False,
                                      note_str_in="exp12 step01 exploration (2009-08-13 -> 2014-12-31 continuous replay, context only)")
print(f"Trials logged:\t{len(exploration_trial_pdf)}\tfirst logged_ts {exploration_trial_pdf['logged_ts'].iloc[0]}\ttest_window_touched: {exploration_trial_pdf['test_window_touched'].unique().tolist()}")

Trials logged:	6	first logged_ts 2026-10-09T14:31:19	test_window_touched: [False]

In [9]:
# CALL FUNCTION TO READ THE TRIAL LOG
trial_log_pdf = read_trial_log_pdf()
# DISPLAY THE NUMBER OF TRIALS PER EXPERIMENT AND STAGE (MULTIPLE-TESTING RECORD)
get_trial_count_pdf(trial_log_pdf)

,experiment,stage,trial_count,config_count,test_touched_count,first_logged_ts,last_logged_ts
0,exp01_minute_entry,multivariate_check,1,1,0,2026-10-06T18:27:48,2026-10-06T18:27:48
1,exp01_minute_entry,signal_check,2,2,0,2026-10-05T15:17:38,2026-10-06T12:46:46
2,exp01_minute_entry,summary,1,1,0,2026-10-05T16:16:30,2026-10-05T16:16:30
3,exp01_minute_entry,validation,540,1,0,2026-10-05T15:36:53,2026-10-05T16:16:18
4,exp02_stop_reentry,exploration,14,1,0,2026-10-05T14:47:50,2026-10-05T14:47:50
5,exp02_stop_reentry,signal_check,1,1,0,2026-10-05T14:48:41,2026-10-05T14:48:41
6,exp02_stop_reentry,summary,1,1,0,2026-10-05T14:50:29,2026-10-05T14:50:29
7,exp02_stop_reentry,validation,792,1,0,2026-10-05T14:49:18,2026-10-05T14:49:48
8,exp03_ath_exit,exploration,15,1,0,2026-10-05T14:55:04,2026-10-05T14:55:04
9,exp03_ath_exit,summary,1,1,0,2026-10-05T14:56:49,2026-10-05T14:56:49
